In [ ]:
#24k-0810
# Task 1

class WarehouseEnvironment:
    def __init__(self):
        self.outcomes = {
            "Zone A": [3, 5],
            "Zone B": [2, 9]
        }

    def get_thief_options(self, robot_move):
        return self.outcomes[robot_move]

class RobotAgent:
    def __init__(self, env):
        self.env = env

    def minimax_decision(self):
        print("States and Moves:")
        best_move = None
        best_value = float('-inf')
        
        for move in self.env.outcomes:
            thief_options = self.env.get_thief_options(move)
            thief_choice = min(thief_options)
            print(f"Robot chooses {move}, Thief options {thief_options}, Thief picks {thief_choice}")

            if thief_choice > best_value:
                best_value = thief_choice
                best_move = move
                
        print(f"\nOptimal Robot Move: {best_move} with value {best_value}")
env = WarehouseEnvironment()
robot = RobotAgent(env)
robot.minimax_decision()

States and Moves:
Robot chooses Zone A, Thief options [3, 5], Thief picks 3
Robot chooses Zone B, Thief options [2, 9], Thief picks 2

Optimal Robot Move: Zone A with value 3


In [ ]:
#Task 2
# (a) Stop: min(6, 7) = 6, Go: min(2, 4) = 2, Turn: min(8, 3) = 3
#     max(6, 2, 3) = 6 → Stop

# (b) MAX(Start) α=-∞ β=∞
# ├─ Stop (MIN) α=-∞ β=∞
# │   ├─ 6 → β = min(∞,6) = 6
# │   └─ 7 → β = min(6,7) = 6
# │   → Stop evaluates to 6 → α = max(-∞,6) = 6
# ├─ Go (MIN) α=6 β=∞
# │   ├─ 2 → β = min(∞,2) = 2
# │   └─ 4 → [PRUNED] (α = 6 ≥ β = 2) 
# │   → Go evaluates to 2 → α = max(6,2) = 6
# └─ Turn (MIN) α=6 β=∞
#     ├─ 8 → β = min(∞,8) = 8
#     └─ 3 → β = min(8,3) = 3
#     → Turn evaluates to 3 → α = max(6,3) = 6

#(c) Pruned Branch: Node 4 because it cannot affect the MAX decision.
#(d) Number of nodes not evaluated due to pruning: 1 (node with value 4)

import math
class Node:
    def __init__(self, value):
        self.value = value
        self.children = []
        self.minmax_value = None

class MinimaxAgent:
    def __init__(self, depth):
        self.depth = depth

    def act(self, node, environment):
        return environment.alpha_beta_search(node, self.depth, -math.inf, math.inf, True)

class Environment:
    def __init__(self, tree):
        self.tree = tree
        self.computed_nodes = []

    def alpha_beta_search(self, node, depth, alpha, beta, maximizing_player=True):
        if depth == 0 or not node.children:
            self.computed_nodes.append(node.value)
            node.minmax_value = node.value
            return node.value

        if maximizing_player:
            value = -math.inf
            for child in node.children:
                value = max(value, self.alpha_beta_search(child, depth-1, alpha, beta, False))
                alpha = max(alpha, value)
                if beta <= alpha:
                    for c in node.children[node.children.index(child)+1:]:
                        print(f"Pruned node: {c.value}")
                    break
            node.minmax_value = value
            return value
        else:
            value = math.inf
            for child in node.children:
                value = min(value, self.alpha_beta_search(child, depth-1, alpha, beta, True))
                beta = min(beta, value)
                if beta <= alpha:
                    for c in node.children[node.children.index(child)+1:]:
                        print(f"Pruned node: {c.value}")
                    break
            node.minmax_value = value
            return value

def run_agent(agent, environment, root):
    optimal_value = agent.act(root, environment)
    print(f"\nOptimal value for root (MAX): {optimal_value}\n")

root = Node("Start")
stop = Node("Stop")
go = Node("Go")
turn = Node("Turn")
root.children = [stop, go, turn]
stop.children = [Node(6), Node(7)]
go.children = [Node(2), Node(4)]
turn.children = [Node(8), Node(3)]

depth = 2
agent = MinimaxAgent(depth)
environment = Environment(root)
run_agent(agent, environment, root)

nodes = [root, stop, go, turn]
for n in nodes:
    print(f"{n.value}: {n.minmax_value}")
print("\nNodes evaluated in Alpha-Beta pruning:")
print(environment.computed_nodes)


Pruned node: 4

Optimal value for root (MAX): 6

Start: 6
Stop: 6
Go: 2
Turn: 3

Nodes evaluated in Alpha-Beta pruning:
[6, 7, 2, 8, 3]


In [4]:
#Task 3
# (a) A1: min(5,6)=5, A2: min(7,4)=4 ---> Attack evaluates to min(5,4)=4
#     D1: min(3,8)=3, D2: min(6,2)=2 ---> Defend evaluates to min(3,2)=2
#     G1: min(1,9)=1, G2: min(4,7)=4 ---> Gather evaluates to min(1,4)=1
#     Root (MAX) chooses: max(4,2,1)=4 → Attack

import math
class Node:
    def __init__(self, value):
        self.value = value
        self.children = []
        self.minmax_value = None
class MinimaxAgent:
    def __init__(self, depth):
        self.depth = depth

    def act(self, node, environment):
        return environment.alpha_beta_search(node, self.depth, -math.inf, math.inf, True)

class Environment:
    def __init__(self, tree):
        self.tree = tree
        self.computed_nodes = []

    def alpha_beta_search(self, node, depth, alpha, beta, maximizing_player=True):
        if depth == 0 or not node.children:
            self.computed_nodes.append(node.value)
            node.minmax_value = node.value
            return node.value

        if maximizing_player:
            value = -math.inf
            for child in node.children:
                value = max(value, self.alpha_beta_search(child, depth-1, alpha, beta, False))
                alpha = max(alpha, value)
                if beta <= alpha:
                    for c in node.children[node.children.index(child)+1:]:
                        print(f"Pruned node: {c.value}")
                    break
            node.minmax_value = value
            return value
        else:
            value = math.inf
            for child in node.children:
                value = min(value, self.alpha_beta_search(child, depth-1, alpha, beta, True))
                beta = min(beta, value)
                if beta <= alpha:
                    for c in node.children[node.children.index(child)+1:]:
                        print(f"Pruned node: {c.value}")
                    break
            node.minmax_value = value
            return value

def run_agent(agent, environment, root):
    optimal_value = agent.act(root, environment)
    print(f"\nOptimal value for root (MAX): {optimal_value}\n")

root = Node("Root")
attack = Node("Attack")
defend = Node("Defend")
gather = Node("Gather")
root.children = [attack, defend, gather]
A1 = Node("A1")
A2 = Node("A2")
attack.children = [A1, A2]
D1 = Node("D1")
D2 = Node("D2")
defend.children = [D1, D2]
G1 = Node("G1")
G2 = Node("G2")
gather.children = [G1, G2]

A1.children = [Node(5), Node(6)]
A2.children = [Node(7), Node(4)]
D1.children = [Node(3), Node(8)]
D2.children = [Node(6), Node(2)]
G1.children = [Node(1), Node(9)]
G2.children = [Node(4), Node(7)]

depth = 3
agent = MinimaxAgent(depth)
environment = Environment(root)
run_agent(agent, environment, root)
print("Minimax values:")
nodes = [root, attack, defend, gather, A1, A2, D1, D2, G1, G2]
for n in nodes:
    print(f"{n.value}: {n.minmax_value}")

print("\nNodes evaluated in Alpha-Beta pruning:")
print(environment.computed_nodes)


Pruned node: 4

Optimal value for root (MAX): 7

Minimax values:
Root: 7
Attack: 6
Defend: 6
Gather: 7
A1: 6
A2: 7
D1: 8
D2: 6
G1: 9
G2: 7

Nodes evaluated in Alpha-Beta pruning:
[5, 6, 7, 3, 8, 6, 2, 1, 9, 4, 7]
